# Paytech SPRINT 4

O objetivo da POC da Sprint 4 da Paytech One é cosenguir mapear elementos de um abastecimento com tecnicas de visão computacional

## Anotação dos Dados

A paytech enviou um conjunto de arquivos contendo varias gravações de abastecimentos de diferentes dias e horarios. Para treinar um modelo com esses dados foi necessario primeiro fazer a anotação correta deles. 

Por se tratar de dado sensiveis, a ferramenta escolhida para a anotação dos dados foi a CVAT, atuando em uma instancia local.

Antes da anotação, o conjunto foi primeito mapedo, para identificar quais arquivos de fato continham abastecimentos.

**5 Classes foram criadas:**
- Pessoa 
- Veiculo 
- Placa
- Pistola
- Totem

## Treino do modelo (Experimento 2)

Uma vez com o conjunto total de dados anotados corretamente e separados em suas respectivas pastas de treino e teste, vamos passar para o treino propriamente dito do YOLO.

As informações do conjunto de dados do dataset se encontra no arquivo `data.yaml`

O treino do Modelo apresentado nesse Jupyter Notebook consiste no segundo Experimento de treino do modelo YOLO. O primeiro experimento foi um treino mais simples, sem muitas epocas e feito atarvez de um Python Script "train.py"  

O treino a seguir visa explorar mais epocas e outras configurações de hiperparametros (consultar o relatório tecnico)

### OBS: 
O experimento foi feito utilizando o processador M5 para aumentar a velocidade de processamento, mas pode ser facilmente reporduzido em outro computador desde que modificado o parametro `device`

In [3]:
from ultralytics import YOLO

# Carrega o modelo (ex: YOLOv8 ou YOLOv11 nano)
model = YOLO('yolov8n.pt')

# Inicia o treino com os hiperparâmetros configurados
results = model.train(
    data='./data.yaml',
    device='mps',      # Força o uso da GPU do chip Apple Silicon M5
    epochs=100,        # Número de passagens completas pelo dataset
    imgsz=640,         # Resolução que a imagem será redimensionada
    batch=16,          # Quantidade de imagens processadas por vez
    lr0=0.01,          # Taxa de aprendizado inicial
    patience=15,       # Early stopping (para o treino se não melhorar)
    workers=4          # Processamento paralelo para carregar dados
)

New https://pypi.org/project/ultralytics/8.4.83 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.82 🚀 Python-3.9.6 torch-2.8.0 MPS (Apple M5)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-3, nbs=64, nms=False, opset=No

: 

Houve uma interrupção do treino do modelo por problemas internos no Kernel, entretando o resultado das 78 epocas das 100 estipuladas foi salvo dentro do diretorio `/run` do projeto. E como é possivel ver na última linha de log, as metricas do modelo ja apresentava um otimo desempenho.

## Explorando resultados de treino e validação do experimento

In [2]:
from ultralytics import YOLO

# 1. Carrega o melhor modelo salvo antes da queda do kernel
model = YOLO('runs/detect/train-4/weights/best.pt')

# 2. Roda a validação isolada usando o seu data.yaml
metrics = model.val(data='data.yaml')

# Opcional: printar algumas métricas na tela
print(f"mAP50 final na validação: {metrics.box.map50}")
print(f"mAP50-95 final na validação: {metrics.box.map}")

Ultralytics 8.4.82 🚀 Python-3.9.6 torch-2.8.0 CPU (Apple M5)
Model summary (fused): 73 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3513.1±676.9 MB/s, size: 4389.2 KB)
val: Scanning /Users/aryel/Documents/paytech/val/labels.cache... 444 images, 25 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 444/444 40.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 28/28 1.5s/it 42.2s1.6ss
                   all        444       1506      0.933      0.954      0.978      0.866
                  Bico        361        381      0.971       0.95      0.991       0.93
                Pessoa        228        264      0.876       0.89      0.936      0.704
               Veiculo        345        346      0.937          1      0.993      0.984
                 Toten        415        415      0.951       0.99      0.993      0.972
                 placa        100        100      